# Example

AI development requires interacting with raw text files—whether loading vast datasets for training or saving evaluation logs. To handle files safely, you must use the with statement (a context manager). It guarantees that Python automatically closes the file stream and releases system memory hooks even if your internal code throws an explicit runtime error.

In [1]:
# Open a text file for safe writing using context handling
# "w" mode overwrites existing content; "r" is for reading
with open("dataset_log.txt", "w", encoding="utf-8") as file_stream:
    # Write explicit strings down to the target path
    file_stream.write("epoch_1: loss=0.452\n")
    file_stream.write("epoch_2: loss=0.211\n")

# File stream is now automatically closed by the interpreter context here

# Re-open the same file to read its contents back sequentially
with open("dataset_log.txt", "r", encoding="utf-8") as file_stream:
    # Read the entire raw text block out at once
    content = file_stream.read()
    print(content)

epoch_1: loss=0.452
epoch_2: loss=0.211



# Exercise

Write a script that opens a file named model_output.txt in append mode ("a"). Write the string "Validation Accuracy: 94.2%\n" into it. Use a context manager to ensure the file stream handles closures cleanly.

In [2]:
with open("model_output.txt", "a") as file:
    file.write("Validation Accuracy: 94.2%\n")

# Example

AI models use configuration files to track settings like learning rates, layer sizes, and token boundaries. JSON (JavaScript Object Notation) is the universal format for these configurations because it maps perfectly to Python dictionaries. The json module provides two key tools: json.dump() (which translates a Python dictionary into a static JSON file) and json.load() (which parses a JSON file back into a live Python dictionary).

In [3]:
import json

# A live model configuration state stored inside a standard dictionary
config_dict = {
    "model_name": "mini-GPT",
    "hidden_dim": 512,
    "pretrained": True
}

# json.dump() translates the live dictionary into a static file stream
with open("hyperparameters.json", "w") as json_file:
    json.dump(config_dict, json_file, indent=4)

# json.load() pulls the static file back into an active dictionary structure
with open("hyperparameters.json", "r") as json_file:
    loaded_dict = json.load(json_file)

print(loaded_dict["hidden_dim"])

512


# Exercise

You have a target dictionary detailing evaluation benchmarks:
eval_metrics = {"loss": 0.042, "accuracy": 0.981, "status": "converged"}

Write a script that exports this dictionary to a file named metrics.json. Then, re-open that file using json.load() and print the value assigned to the "accuracy" key.

In [6]:
eval_metrics = {"loss": 0.042, "accuracy": 0.981, "status": "converged"}

with open("metrics.json", "w") as json_file:
    json.dump(eval_metrics, json_file, indent=4)

with open("metrics.json", "r") as json_file:
    loaded_dict = json.load(json_file)
print(loaded_dict["accuracy"])

0.981


# Example

Hardcoding raw strings for file paths (like "C:\data\tokens.txt" or "/data/tokens.txt") makes your AI code brittle and likely to crash when shared across different operating systems (Windows uses backslashes \, while Linux uses forward slashes /). Python's modern pathlib module solves this by treating paths as objects rather than strings, ensuring cross-platform compatibility out of the box.

In [7]:
from pathlib import Path

# Create a platform-agnostic path object
# It automatically formats slashes correctly for Windows or Linux
data_dir = Path("datasets") / "text_corpus"
print(f"Path Object: {data_dir}") # Output on Linux: datasets/text_corpus

# Check file system status directly using Path methods
print(f"Exists? {data_dir.exists()}")
print(f"Is a folder? {data_dir.is_dir()}")

# Grab the filename or extension details instantly
target_file = Path("models/checkpoint.pt")
print(f"Suffix: {target_file.suffix}") # Output: .pt
print(f"Name: {target_file.name}")     # Output: checkpoint.pt

Path Object: datasets/text_corpus
Exists? False
Is a folder? False
Suffix: .pt
Name: checkpoint.pt


# Exercise

Using the pathlib library, create a path object that points to a nested structure: a folder named checkpoints containing a file named weights.bin. Print the path object, and then use its .suffix property to print just the file extension.

In [9]:
from pathlib import Path


path = Path("checkpoints") / "weights.bin"

print(path)
print(path.suffix)

checkpoints/weights.bin
.bin


# Example

When loading raw files for text models, you will often find yourself dealing with large corpora that cannot fit entirely into your computer's RAM. Instead of using .read() to dump the entire file into memory at once, you can iterate over the file stream object using a loop. This reads the file line by line, processing small chunks of data sequentially while keeping memory consumption low and stable.

In [10]:
# Create a dummy multiline corpus file
with open("corpus.txt", "w") as f:
    f.write("First text sequence.\nSecond text sequence.\nThird text sequence.")

# Line-by-line streaming configuration
with open("corpus.txt", "r") as file_stream:
    # This loop reads one line into RAM at a time, then swaps it out
    for line in file_stream:
        # Strip trailing newline characters '\n'
        clean_line = line.strip()
        print(f"Processed line: {clean_line}")

Processed line: First text sequence.
Processed line: Second text sequence.
Processed line: Third text sequence.


# Exercise

Write a script that reads a text file named logs.txt line by line. If a line contains the word "ERROR", print that line. If it does not contain "ERROR", ignore it.

In [11]:
with open("logs.txt", "r") as file:
    for line in file:
        if "ERROR" in line:
            print(line, end="")

FileNotFoundError: [Errno 2] No such file or directory: 'logs.txt'

# FINAL EXERCISE

Objective:
Build an automated training metrics logger and parser. In deep learning frameworks, you frequently parse raw text output logs to extract performance metrics, which you then export into standardized JSON files for visualization tools like TensorBoard.

Requirements:
1. Setup: Create a dummy log file named raw_training.log and populate it with the following multiline string:
[INFO] Epoch 1 starting...
[METRIC] loss: 0.852, accuracy: 0.612
[INFO] Epoch 2 starting...
[METRIC] loss: 0.412, accuracy: 0.824
[INFO] Epoch 3 starting...
[METRIC] loss: 0.154, accuracy: 0.941
2. Streaming & Parsing: Open raw_training.log and read it line by line.
3. Extraction: Find the lines starting with [METRIC]. Extract the raw loss value (as a float) and accuracy value (as a float) from those lines.
4. Data Aggregation: Store these extracted metrics inside a dictionary structured exactly like this:
parsed_metrics = {
    "loss_history": [0.852, 0.412, 0.154],
    "accuracy_history": [0.612, 0.824, 0.941]
}
5. JSON Export: Use pathlib.Path to target an export path named output_metrics.json. Write the parsed_metrics dictionary into that file layout using the json module with an indentation alignment of 4 spaces.
6. Verification: Read the exported output_metrics.json file back into memory and print its contents to prove the export succeeded.

Prohibited:
Do not use external parsing libraries like pandas or regex. Stick strictly to standard Python string manipulation methods (like .startswith(), .split(), .strip()).

In [13]:
from pathlib import Path
import json

log_contents = """[INFO] Epoch 1 starting...
[METRIC] loss: 0.852, accuracy: 0.612
[INFO] Epoch 2 starting...
[METRIC] loss: 0.412, accuracy: 0.824
[INFO] Epoch 3 starting...
[METRIC] loss: 0.154, accuracy: 0.941
"""

with open("raw_training.log", "w") as log_file:
    log_file.write(log_contents)

# 2-4. Read, parse, and aggregate metrics
parsed_metrics = {
    "loss_history": [],
    "accuracy_history": []
}

with open("raw_training.log", "r") as log_file:
    for line in log_file:
        line = line.strip()

        if line.startswith("[METRIC]"):
            # Remove the prefix
            metric_data = line.replace("[METRIC] ", "")

            # Split into loss and accuracy components
            parts = metric_data.split(", ")

            loss_value = float(parts[0].split(": ")[1])
            accuracy_value = float(parts[1].split(": ")[1])

            parsed_metrics["loss_history"].append(loss_value)
            parsed_metrics["accuracy_history"].append(accuracy_value)

# 5. Export to JSON using pathlib
output_path = Path("output_metrics.json")

with output_path.open("w") as json_file:
    json.dump(parsed_metrics, json_file, indent=4)

# 6. Read the JSON back and print its contents
with output_path.open("r") as json_file:
    loaded_metrics = json.load(json_file)

print(loaded_metrics)

{'loss_history': [0.852, 0.412, 0.154], 'accuracy_history': [0.612, 0.824, 0.941]}


In [14]:
output_path = Path("output_metrics.json")

with output_path.open("w") as json_file:
    json.dump(parsed_metrics, json_file, indent=4)

with output_path.open("r") as json_file:
    loaded_metrics = json.load(json_file)

print(loaded_metrics)

{'loss_history': [0.852, 0.412, 0.154], 'accuracy_history': [0.612, 0.824, 0.941]}
